# Tutorial

This tutorial walks through the core workflow of `pastax`:

1. Build a synthetic forcing made of an ocean surface jet and an overlying wind, wrap it in a `Dataset`.
2. Deterministic trajectory simulation
   1. Run a **deterministic** trajectory through the field with the `solve` ODE integrator, including a windage term.
   2. **Learn** the windage coefficient by JAX automatic differentiation, using `optimistix` and the separation distance as a residual.
3. Stochastic trajectory ensemble simulation
   1. Run a **stochastic** ensemble using a Smagorinsky-style diffusion on a **smoothed** copy of the current — exercising the SDE mode of `solve` and the `Dataset.neighborhood` API.
   2. **Jointly learn** the windage and the Smagorinsky coefficient on a stochastic simulator, using a time-aggregated energy score with separation distance as kernel.
4. More "advanced" modelisation
   1. Run a **perturbed-ODE** ensemble with a non-linear tanh-squashed noise residual — illustrating the ODE+controls pattern for MDN / flow-matching style simulators.
   2. Run a **second-order** SDE with the full surface-ocean **Maxey–Riley** equation (a second-order PyTree `(position, velocity)` state, Coriolis, fluid acceleration and a wind/water-weighted carrier), perturbed by anisotropic eddy-diffusivity turbulence built as a `lineax` operator.

The same pattern transposes to real forcing fields loaded from xarray (zarr / netCDF) via `Dataset.from_xarray`.

> Boilerplate cells (imports, plot setup, animation rendering) are folded by default — click *Show code* to expand. The cells most relevant to the `pastax` API are always expanded.

In [ ]:
import cmocean
from IPython.display import HTML
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import optimistix as optx


## 1. Defining the forcing fields

We build two synthetic fields on the same grid:

- a meandering zonal **ocean jet** with circular eddies and some small scale noise, 
  giving a realistic-looking surface current $\mathbf{u}_o$;
- a slowly **rotating wind** $\mathbf{u}_w$ — spatially uniform, time-varying, deliberately
  decorrelated from the ocean current.

Both fields share the same grid (101 × 101 × 97), spanning two degrees in lat/lon and four days in time.

In [ ]:
from pastax import degrees_to_meters

NY = NX = 101
NT = 97  # 4 days, hourly

LAT = jnp.linspace(29.0, 31.0, NY)
LON = jnp.linspace(-1.0, 1.0, NX)

DT = 60 * 60                                # one hour, in seconds
TS = jnp.linspace(0.0, DT * (NT - 1), NT)   # 0 .. 96 h

DLAT = float(LAT[1] - LAT[0])               # grid steps for finite differences
DLON = float(LON[1] - LON[0])
DY_M, DX_M = degrees_to_meters(jnp.asarray([DLAT, DLON]), LAT.mean())


In [ ]:
# ---- coordinate grids in meters ----
y_m = jnp.arange(NY) * DY_M
x_m = jnp.arange(NX) * DX_M
X, Y = jnp.meshgrid(x_m, y_m)         # (NY, NX)
Lx, Ly = x_m[-1], y_m[-1]
T_total = float(TS[-1])

# =====================================================================
# 1. Meandering jet with TWO superposed meander modes
# =====================================================================
U0          = 0.6                           # peak jet speed [m/s]
L_jet       = 18e3                          # jet half-width [m]
y_jet       = 0.5 * Ly
A1, k1, c1  = 10e3, 2*jnp.pi/(0.55*Lx), 0.05      # primary meander
A2, k2, c2  = 5e3, 2*jnp.pi/(0.25*Lx), -0.03     # smaller, opposite-going

def jet_psi(x, y, t):
    eta = (y - y_jet
           - A1 * jnp.sin(k1 * (x - c1 * t))
           - A2 * jnp.sin(k2 * (x - c2 * t)))
    return -U0 * L_jet * jnp.tanh(eta / L_jet)

# =====================================================================
# 2. Many small eddies, each with drifting+oscillating centre and a
#    smooth amplitude envelope (so eddies appear and fade)
# =====================================================================
key = jax.random.PRNGKey(2)
n_eddies = 22
ks = jax.random.split(key, 10)

# Centres uniform across the domain
x0s    = jax.random.uniform(ks[0], (n_eddies,), minval=0.05*Lx, maxval=0.95*Lx)
y0s    = jax.random.uniform(ks[1], (n_eddies,), minval=0.10*Ly, maxval=0.90*Ly)

# Sizes: small eddies, 6-13 km
sigmas = jax.random.uniform(ks[2], (n_eddies,), minval=6e3, maxval=13e3)

# Mixed cyclonic / anticyclonic.  peak swirl ≈ 0.606 * |amp| / sigma
signs  = jax.random.choice(ks[3], jnp.array([-1.0, 1.0]), (n_eddies,))
mags   = jax.random.uniform(ks[4], (n_eddies,), minval=4e3, maxval=9e3)
amps   = signs * mags

# Linear drift (slow, m/s)
ud     = jax.random.uniform(ks[5], (n_eddies,), minval=-0.025, maxval=0.025)
vd     = jax.random.uniform(ks[6], (n_eddies,), minval=-0.020, maxval=0.020)

# Position oscillation: small loop on top of the drift
osc_amp   = jax.random.uniform(ks[7], (n_eddies,), minval=2e3, maxval=6e3)
osc_omega = 2*jnp.pi / jax.random.uniform(ks[8], (n_eddies,),
                                          minval=1.0*86400, maxval=3.0*86400)
osc_phase = jax.random.uniform(ks[9], (n_eddies,), minval=0.0, maxval=2*jnp.pi)

# Amplitude envelope: each eddy peaks at a random time, with random width
key2 = jax.random.PRNGKey(11)
ks2  = jax.random.split(key2, 2)
env_t0    = jax.random.uniform(ks2[0], (n_eddies,), minval=-0.5*T_total,
                                maxval=1.5*T_total)
env_sigma = jax.random.uniform(ks2[1], (n_eddies,), minval=1.5*86400,
                                maxval=4.0*86400)

def eddies_psi(x, y, t):
    """ψ-contribution from all eddies. x,y: (NY,NX). t: scalar. -> (NY,NX)"""
    # time-varying centres (n_eddies,)
    xc = x0s + ud*t + osc_amp * jnp.cos(osc_omega*t + osc_phase)
    yc = y0s + vd*t + osc_amp * jnp.sin(osc_omega*t + osc_phase)  # circular loop
    # smooth envelope: gaussian pulse in time
    env   = jnp.exp(-((t - env_t0) / env_sigma)**2)
    amp_t = amps * env                                              # (n_eddies,)
    # broadcast: (NY, NX, 1) - (1, 1, n_eddies)  ->  (NY, NX, n_eddies)
    dx2 = (x[..., None] - xc)**2
    dy2 = (y[..., None] - yc)**2
    inv_2s2 = 1.0 / (2.0 * sigmas**2)
    return jnp.sum(amp_t * jnp.exp(-(dx2 + dy2) * inv_2s2), axis=-1)

# =====================================================================
# 3. Sub-mesoscale turbulence via 2D Navier-Stokes (exponax)
#    Replaces the ad-hoc bandpass noise with a vorticity field evolved
#    under the 2D incompressible NS equations, giving filamentary texture
#    that is physically consistent with a turbulent cascade.
# =====================================================================
import exponax as ex

NS_DT     = 0.05   # NS timestep (non-dimensional)
NS_SPINUP = 500    # steps discarded to reach a turbulent state

ns_stepper = ex.stepper.NavierStokesVorticity(
    num_spatial_dims=2,
    domain_extent=2 * jnp.pi,
    num_points=NX,        # 101 — matches the tutorial grid exactly
    dt=NS_DT,
    diffusivity=0.001,    # low viscosity → more filamentary texture
)

omega0 = ex.ic.RandomTruncatedFourierSeries(
    num_spatial_dims=2, cutoff=5
)(num_points=NX, key=jax.random.PRNGKey(33))   # (1, NX, NX)

# Spin up, then produce NT frames — one per tutorial timestep.
# The NS time is non-dimensional; only the evolving spatial structure matters.
omega_spinup = jax.lax.fori_loop(0, NS_SPINUP, lambda _, w: ns_stepper(w), omega0)
omega_traj   = ex.rollout(ns_stepper, NT, include_init=False)(omega_spinup)
# shape: (NT, 1, NX, NX)

def _omega_to_psi(omega_frame):
    """Invert Δψ = ω in spectral space: ψ̂ = ω̂ / k²  (k=0 mode pinned to 0)."""
    om = omega_frame[0]                          # (NX, NX)
    N  = om.shape[0]
    ki = jnp.fft.fftfreq(N) * N                 # integer wavenumbers
    KI, KJ = jnp.meshgrid(ki, ki, indexing='ij')
    K2 = KI**2 + KJ**2
    psi_hat = jnp.where(K2 == 0, 0.0, jnp.fft.fft2(om) / K2)
    return jnp.real(jnp.fft.ifft2(psi_hat))

psi_ns = jax.vmap(_omega_to_psi)(omega_traj)    # (NT, NX, NX)

NOISE_AMP_NS = 1e2                               # stream-function amplitude [m²/s]
psi_ns = psi_ns / psi_ns.std() * NOISE_AMP_NS

# =====================================================================
# 4. Total stream function and velocity
# =====================================================================
def stream_function(x, y, t):
    return jet_psi(x, y, t) + eddies_psi(x, y, t)

# Deterministic jet + eddies (vmapped over time), then superpose NS texture.
psi_det = jax.vmap(lambda t: stream_function(X, Y, t))(TS)   # (NT, NY, NX)
psi = psi_det + psi_ns

u_o = -jnp.gradient(psi, DY_M, axis=1)
v_o =  jnp.gradient(psi, DX_M, axis=2)

print("u_o, v_o shape:", u_o.shape,
      f" peak speed: {float(jnp.sqrt(u_o ** 2 + v_o ** 2).max()):.2f} m/s",
      f" min speed: {float(jnp.sqrt(u_o ** 2 + v_o ** 2).min()):.2f} m/s")

In [ ]:
# Spatially-uniform wind with rotating direction and a 3-day modulation in magnitude
W_MEAN  = 8.0                               # mean wind speed (m/s)
W_AMP   = 4.0                               # peak-to-mean magnitude variation (m/s)
T_wind  = 4.0 * 86400.0                     # full rotation in 4 days
T_wamp  = 3.0 * 86400.0                     # magnitude modulation period (s)
alpha0  = jnp.pi / 4                        # initial direction (NE)

def wind_uv(t):
    alpha = alpha0 + 2 * jnp.pi * t / T_wind
    W_t   = W_MEAN + W_AMP * jnp.sin(2 * jnp.pi * t / T_wamp)
    return (W_t * jnp.cos(alpha) * jnp.ones((NY, NX)),
            W_t * jnp.sin(alpha) * jnp.ones((NY, NX)))

u_w, v_w = jax.vmap(wind_uv)(TS)

print("u_w, v_w shape:", u_w.shape)
print("||u_w|| range:", 
      f"{float(jnp.sqrt(u_w ** 2 + v_w ** 2).min()):.2f} - "
      f"{float(jnp.sqrt(u_w ** 2 + v_w ** 2).max()):.2f} m/s")


Animation of the joint forcing — the **ocean current speed** is shown in colour with white arrows for the current direction; the **wind** is overlaid as black arrows. 
The jet meanders, the background drift is uniform, and the wind both rotates and modulates in magnitude.

In [ ]:
ocean_speed = jnp.sqrt(u_o ** 2 + v_o ** 2)
ocean_clim = float(ocean_speed.max())

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

im = ax.pcolormesh(LON, LAT, ocean_speed[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim)
q_o = ax.quiver(LON[::6], LAT[::6], u_o[0][::6, ::6], v_o[0][::6, ::6],
                scale=20, color="white", alpha=0.8, width=0.004)
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.9, width=0.006,
                pivot="mid")
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$)")
ax.set_aspect("equal")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
title = ax.set_title(f"$t$ = {TS[0] / 3600:.0f} h")

def draw(k):
    im.set_array(np.asarray(ocean_speed[k]).ravel())
    q_o.set_UVC(np.asarray(u_o[k][::6, ::6]), np.asarray(v_o[k][::6, ::6]))
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    title.set_text(f"$t$ = {TS[k] / 3600:.0f} h")
    return im, q_o, q_w, title

HTML(animation.FuncAnimation(fig, draw, frames=NT, interval=80, blit=False).to_jshtml())


Wrap both fields into a single `Dataset` so they can be queried by the integrator.
`Dataset.from_arrays` accepts plain numpy or JAX arrays directly; for real forcings, use `Dataset.from_xarray` instead.

In [ ]:
from pastax import Dataset

forcing = Dataset.from_arrays(
    fields={"u_o": u_o, "v_o": v_o, "u_w": u_w, "v_w": v_w},
    t=TS,
    lat=LAT,
    lon=LON,
    dtype=jnp.float64,
)


## 2. Deterministic trajectory simulation

### 1. A deterministic trajectory with windage

A surface object is advected by the ocean current and partially dragged by the wind. The **direct windage** model parameterises this as

$$
\mathrm{d}\mathbf{X}(t) = \bigl[\mathbf{u}_o(t, \mathbf{X}) + \beta_w \, \mathbf{u}_w(t, \mathbf{X})\bigr] \, \mathrm{d}t,
$$

with $\beta_w$ a small dimensionless coefficient — typically $0.1$ to $10\%$ depending on the object. 
Here we take $\beta_w = 3\%$ as ground truth.

In `pastax` the dynamics are expressed as a Python callable `term(t, y, args)` that returns the velocity in degrees per second. 
`solve` defaults to ODE mode and uses the `Tsit5` integrator with a fixed step size.

In [ ]:
from pastax import solve, Tsit5, meters_to_degrees

BETA_W_TRUE = 0.03  # 3% direct windage

def windage_term(t, y, args):
    dataset, beta_w = args
    lat_, lon_ = y[0], y[1]
    uo = dataset["u_o"].interp(t, lat_, lon_)
    vo = dataset["v_o"].interp(t, lat_, lon_)
    uw = dataset["u_w"].interp(t, lat_, lon_)
    vw = dataset["v_w"].interp(t, lat_, lon_)
    u = uo + beta_w * uw
    v = vo + beta_w * vw
    # convert (north=v, east=u) m/s -> (dlat/dt, dlon/dt) deg/s
    return meters_to_degrees(jnp.array([v, u]), lat_)

y0 = jnp.array([30, -0.75])           # initial [lat, lon] -- just south of jet axis
ts_sim = TS[1:-1]                      # kept for plotting / indexing

# Derive solver parameters from ts_sim (avoids boundary timestamps)
t0_sim     = float(ts_sim[0])
n_save_sim = len(ts_sim) - 1
int_dt_sim = float(ts_sim[1] - ts_sim[0])

traj = solve(windage_term, y0,
             jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Tsit5(), 
             args=(forcing, BETA_W_TRUE))
print("trajectory shape:", traj.shape)

Animation — the time-evolving ocean speed and wind velocity underneath, the trajectory growing one step at a time.

In [ ]:
traj_np = np.asarray(traj)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

im = ax.pcolormesh(LON, LAT, ocean_speed[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim)
fig.colorbar(im, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$)")
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.9, width=0.006,
                pivot="mid")
line, = ax.plot([], [], color="tab:orange", lw=2)
head = ax.scatter([], [], color="white", edgecolor="black", zorder=3)
ax.set_aspect("equal")
ax.set_xlim(float(LON.min()), float(LON.max()))
ax.set_ylim(float(LAT.min()), float(LAT.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
title = ax.set_title("")

# match field timestep to trajectory timestep (ts_sim starts at index 1)
def draw(k):
    field_k = k + 1
    im.set_array(np.asarray(ocean_speed[field_k]).ravel())
    line.set_data(traj_np[: k + 1, 1], traj_np[: k + 1, 0])
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    head.set_offsets(np.asarray([[traj_np[k, 1], traj_np[k, 0]]]))
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return im, line, q_w, head, title

HTML(animation.FuncAnimation(fig, draw, frames=len(traj_np), interval=80, blit=False).to_jshtml())


### 2. Learning the windage coefficient

`pastax`'s `solve` is fully differentiable. 
We exploit that to **fit** the parameters of a term function: given a reference trajectory $\mathbf{Y}^*$ produced by the *true* dynamics, recover the parameters of a model that matches it.

We start with the deterministic case: recover the windage coefficient $\beta_w$ from a single trajectory, using the per-time **separation distance** between simulated and reference paths as residuals.
The model uses the (perfectly observed) true ocean and wind fields but a tunable $\beta_w$; the reference trajectory is generated with $\beta_w = 3\%$.

In [ ]:
class TunableWindage(eqx.Module):
    beta_w: jax.Array

    def __call__(self, t, y, args):
        dataset = args
        return windage_term(t, y, (dataset, self.beta_w))
    
    def get_physical_beta_w(self):
        return self.beta_w

term_init = TunableWindage(beta_w=jnp.array(0.015))

# Reference trajectory: true windage on the true ocean+wind dataset
true_traj = solve(windage_term, y0,
                  jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Tsit5(),
                  args=(forcing, BETA_W_TRUE))

# Initial estimate: tunable term with beta_w = 0
init_traj = solve(term_init, y0,
                  jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Tsit5(),
                  args=forcing)
print("ref shape:", true_traj.shape, "  init shape:", init_traj.shape)

In [ ]:
from pastax import Heun, separation_distance

@eqx.filter_jit
def residual_fn(term_module, ref_traj):
    # Levenberg-Marquardt builds the residual Jacobian in forward mode (jvp), so we
    # opt into adjoint="forward" — the default "checkpointed" adjoint is reverse-mode only.
    est_traj = solve(term_module, y0,
                     jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Heun(),
                     args=forcing, adjoint="forward")
    return separation_distance(est_traj, ref_traj)  # (T,) residuals in metres

solver = optx.BestSoFarLeastSquares(optx.LevenbergMarquardt(rtol=1e-4, atol=1e-4))
sol = optx.least_squares(residual_fn, solver=solver, y0=term_init, args=true_traj)
term_fit = sol.value
print(f"converged in {int(sol.stats['num_steps'])} steps  ->  "
      f"beta_w = {float(term_fit.get_physical_beta_w()) * 100:.1f}%")
print(f"truth                  ->  beta_w = {BETA_W_TRUE * 100:.1f}%")

Animation — the truth, the initial guess, and the fitted trajectory drawn together over the time-evolving ocean speed and wind velocity.

In [ ]:
final_traj = solve(term_fit, y0,
                   jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Tsit5(),
                   args=forcing)
truth_np_d = np.asarray(true_traj)
init_np_d  = np.asarray(init_traj)
final_np_d = np.asarray(final_traj)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

bg = ax.pcolormesh(LON, LAT, ocean_speed[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.7)
fig.colorbar(bg, ax=ax, label=r"$\| \mathbf{u}_o \|$  (m s$^{-1}$)")
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.7, width=0.006,
                pivot="mid")

l_true,  = ax.plot([], [], color="tab:blue",   lw=2.0,             label="Truth")
l_init,  = ax.plot([], [], color="tab:red",    lw=1.5, ls="--",    label=r"Initial ($\beta_w = $" + f"{float(term_init.get_physical_beta_w()) * 100:.1f}%" + r"$)$")
l_final, = ax.plot([], [], color="tab:orange", lw=2.0, ls=":",     label="Fitted")
ax.scatter(y0[1], y0[0], color="white", edgecolor="black", zorder=4, label="Start")
ax.set_aspect("equal")
ax.set_xlim(float(LON.min()), float(LON.max()))
ax.set_ylim(float(LAT.min()), float(LAT.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
title = ax.set_title("")

def draw(k):
    field_k = k + 1
    bg.set_array(np.asarray(ocean_speed[field_k]).ravel())
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    l_true.set_data(truth_np_d[: k + 1, 1],  truth_np_d[: k + 1, 0])
    l_init.set_data(init_np_d[: k + 1, 1],   init_np_d[: k + 1, 0])
    l_final.set_data(final_np_d[: k + 1, 1], final_np_d[: k + 1, 0])
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return bg, q_w, l_true, l_init, l_final, title

HTML(animation.FuncAnimation(fig, draw, frames=len(truth_np_d), interval=80, blit=False).to_jshtml())

## 3. Stochastic trajectory ensemble simulation

### 1. A stochastic ensemble with Smagorinsky diffusion

In a real setting the gridded ocean current we feed the simulator is **smoothed** relative to the actual flow a drifter feels -- think of altimetry-derived surface currents, which only resolve the mesoscale. 
From here on the model therefore runs on a **smoothed copy** of $\mathbf{u}_o$ -- the small-scale filaments of §1 removed -- and the role of a stochastic term is to compensate for the small-scale features the gridded field misses.

In [ ]:
def stream_function_smooth(x, y, t):
    return jet_psi(x, y, t) + eddies_psi(x, y, t)  # drop the filamentary noise -> smoother fields

# build over time. vmap is the cleanest way for the eddy sum.
psi_smooth = jax.vmap(lambda t: stream_function_smooth(X, Y, t))(TS)   # (NT, NY, NX)

u_o_smooth = -jnp.gradient(psi_smooth, DY_M, axis=1)
v_o_smooth = jnp.gradient(psi_smooth, DX_M, axis=2)

forcing_smooth = Dataset.from_arrays(
    fields={"u_o": u_o_smooth, "v_o": v_o_smooth, "u_w": u_w, "v_w": v_w},
    t=TS,
    lat=LAT,
    lon=LON,
    dtype=jnp.float64,
)
ocean_speed_smooth = np.asarray(jnp.sqrt(u_o_smooth ** 2 + v_o_smooth ** 2))

# Deterministic windage on the smoothed currents: the drift the ensemble spreads around.
traj_smooth = solve(windage_term, y0,
                    jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Tsit5(),
                    args=(forcing_smooth, BETA_W_TRUE))
traj_smooth_np = np.asarray(traj_smooth)

print("smoothing reduced peak ocean speed from "
      f"{float(jnp.sqrt(u_o ** 2 + v_o ** 2).max()):.2f} to "
      f"{float(jnp.sqrt(u_o_smooth ** 2 + v_o_smooth ** 2).max()):.2f} m/s")

Sub-grid turbulence the smoothed $\mathbf{u}_o$ does not resolve is conventionally reintroduced as a stochastic diffusion with a Smagorinsky-style local turbulent viscosity:

$$
K(\mathbf{x}, t) = C_S \, \Delta x^2 \sqrt{2(\partial_x u)^2 + 2(\partial_y v)^2 + (\partial_y u + \partial_x v)^2},
$$

estimated from a $3 \times 3$ patch of the smoothed current via `Dataset.neighborhood`. 
With the solver's Wiener-increment convention $\mathrm{d}\mathbf{W} = \sqrt{\Delta t}\,\mathbf{z}$ and $\mathbf{z} \sim \mathcal{N}(0, I_2)$, the diffusion coefficient that produces a per-step displacement variance of $2K\,\Delta t$ is $\sigma = \sqrt{2K}$ — the $\Delta t$ factor lives in the increment, not in the coefficient.

We switch to SDE mode by passing `key` to `solve`. In SDE mode the term has the signature `term(t, y, args) -> (drift, g)`: it returns the deterministic velocity and the diffusion coefficient $g(t, y)$ separately. 
The solver draws $\mathbf{z} \sim \mathcal{N}(0, I_2)$ and applies $\mathbf{g}\,\mathrm{d}\mathbf{W}$ internally — the term never receives $\mathbf{z}$. 
An ensemble of 100 independent realisations is obtained by passing `n_samples=100`; `solve` splits the key internally.
We use the Stratonovich `Heun` solver — its predictor–corrector reuses the same Wiener increment across both stages.

In [ ]:
from pastax import Heun, safe_sqrt

CS = 0.1  # Smagorinsky coefficient

def smag_windage_term(t, y, args):
    dataset, beta_w, c_s = args
    lat_, lon_ = y[0], y[1]

    uo = dataset["u_o"].interp(t, lat_, lon_)
    vo = dataset["v_o"].interp(t, lat_, lon_)
    uw = dataset["u_w"].interp(t, lat_, lon_)
    vw = dataset["v_w"].interp(t, lat_, lon_)
    u = uo + beta_w * uw
    v = vo + beta_w * vw
    drift = meters_to_degrees(jnp.array([v, u]), lat_)

    patches = dataset.neighborhood(t, lat_, lon_, t_window=0, lat_window=1, lon_window=1)
    u_p = patches["u_o"][0]
    v_p = patches["v_o"][0]
    du_dx = (u_p[1, 2] - u_p[1, 0]) / (2 * DX_M)
    du_dy = (u_p[2, 1] - u_p[0, 1]) / (2 * DY_M)
    dv_dx = (v_p[1, 2] - v_p[1, 0]) / (2 * DX_M)
    dv_dy = (v_p[2, 1] - v_p[0, 1]) / (2 * DY_M)

    strain = safe_sqrt(2 * du_dx ** 2 + 2 * dv_dy ** 2 + (du_dy + dv_dx) ** 2)
    K = c_s * DX_M ** 2 * strain
    sigma = safe_sqrt(2 * K)
    g = meters_to_degrees(jnp.array([sigma, sigma]), lat_)

    return drift, g

# n_samples=100 splits the key internally and returns shape (100, n_save+1, 2).
ensemble = solve(smag_windage_term, y0,
                 jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Heun(),
                 args=(forcing_smooth, BETA_W_TRUE, CS),
                 key=jr.key(0), n_samples=100)
print("ensemble shape:", ensemble.shape)

Animation — each thin black line is one stochastic realisation; the orange line is the deterministic, diffusion-free windage trajectory on the same smoothed currents. Both are drawn over the time-evolving smoothed ocean speed and wind velocity.

In [ ]:
ens_np = np.asarray(ensemble)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

bg = ax.pcolormesh(LON, LAT, ocean_speed_smooth[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.7)
fig.colorbar(bg, ax=ax, label=r"$\| \mathbf{u}_o^{\mathrm{smooth}} \|$  (m s$^{-1}$)")
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.7, width=0.006,
                pivot="mid")

ens_lines = [ax.plot([], [], color="black", alpha=0.18, lw=0.6)[0]
             for _ in range(ens_np.shape[0])]
det_line, = ax.plot([], [], color="tab:orange", lw=2.0, label="Deterministic")
ax.scatter(y0[1], y0[0], color="white", edgecolor="black", zorder=4, label="Start")
ax.set_aspect("equal")
ax.set_xlim(float(LON.min()), float(LON.max()))
ax.set_ylim(float(LAT.min()), float(LAT.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
title = ax.set_title("")

def draw(k):
    field_k = k + 1
    bg.set_array(np.asarray(ocean_speed_smooth[field_k]).ravel())
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    for i, ln in enumerate(ens_lines):
        ln.set_data(ens_np[i, : k + 1, 1], ens_np[i, : k + 1, 0])
    det_line.set_data(traj_smooth_np[: k + 1, 1], traj_smooth_np[: k + 1, 0])
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return [bg, q_w, det_line, title, *ens_lines]

HTML(animation.FuncAnimation(fig, draw, frames=ens_np.shape[1], interval=80, blit=False).to_jshtml())


### 2. Jointly learning windage and diffusion

We now move to the **stochastic** setting, building on the smoothed currents introduced in §3.1 -- the only ocean current the model sees. 
The Smagorinsky diffusion stands in for the small-scale features the gridded field misses, and here we recover $(\beta_w, C_S)$ jointly.

We set this up by:

* using the **smoothed currents** of §3.1 (`forcing_smooth`) as the observed forcing;
* generating **10 deterministic reference trajectories** with leeway $\beta_w^* = 3\%$ on the *unsmoothed* currents, seeded in the left side of the domain. 
  Each one is treated as a single sample from the true distribution (the distribution induced by the unresolved scales we are about to parameterise away);
* fitting the model -- which uses the smoothed currents plus a Smagorinsky SDE -- by minimising the time-aggregated **energy score** with separation distance as kernel,
  summed over all 10 reference trajectories:

$$
\mathrm{ES}_t(F, y^*_t) = \frac{1}{M} \sum_{i=1}^M d(X^{(i)}_t, y^*_t) \;-\; \frac{1}{2M(M-1)} \sum_{i \neq j} d(X^{(i)}_t, X^{(j)}_t),
\qquad \mathcal{L} = \sum_{k=1}^{K=10} \sum_t \mathrm{ES}_t(F_n, y^{*,k}_t).
$$

We *do not* have a ground-truth $C_S$ -- $C_S$ is a parameterisation knob, not a physical constant of the unsmoothed flow. 
We start from a non-zero value and ask the optimiser to move it without collapsing it to zero.

> The energy score, alongside the squared error, Dawid-Sebastiani score, and variogram score, ships in `pastax.score` as a proper scoring rule for ensemble forecasts.
> Each accepts a `reduce={None, "last", "sum"}` argument and per-time `weights`; `squared_error` and `energy_score` additionally accept any **broadcasting** distance `kernel` -- here we plug in `pastax.separation_distance` for great-circle distances on the sphere.


In [ ]:
# --- 10 reference trajectories seeded randomly on the left part of the domain ---
N_REFERENCES = 10
y0s_ref = jax.random.uniform(jr.key(42), (N_REFERENCES, 2),
                            minval=jnp.array([LAT[25], LON[5]]),
                            maxval=jnp.array([LAT[-26], LON[50]]))

# Reference trajectories: deterministic windage on the *true* (unsmoothed) currents.
ref_trajs = jax.vmap(
    lambda y0_i: solve(windage_term, y0_i,
                       jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Heun(),
                       args=(forcing, BETA_W_TRUE))
)(y0s_ref)
print("ref_trajs shape:", ref_trajs.shape)

# --- Tunable model: smoothed currents + Smagorinsky SDE ---
class TunableStoch(eqx.Module):
    beta_w_pct: jax.Array  # beta_w stored in percent (O(1))
    c_s_x10:    jax.Array  # c_s   stored *10        (O(1))

    @property
    def beta_w(self):
        return self.beta_w_pct * 0.01
    @property
    def c_s(self):
        return self.c_s_x10 * 0.1

    def __call__(self, t, y, args):
        dataset = args
        return smag_windage_term(t, y, (dataset, self.beta_w, self.c_s))

# Initial guess: a non-zero C_S. We have no ground truth for it; the optimiser
# should move it without collapsing it to zero.
stoch_init = TunableStoch(beta_w_pct=jnp.array(1.5), c_s_x10=jnp.array(1.0))

In [ ]:
from pastax import energy_score

# `energy_score(ens, ref, kernel=separation_distance, reduce="sum")` returns the
# time-aggregated energy score with great-circle distance as kernel:
#   ES = sum_t [ mean_m d(X^m, y_t)  -  sum_{i!=j} d(X^i, X^j) / (2 M (M-1)) ]


In [ ]:
M = 10  # ensemble members per trajectory

@eqx.filter_jit
def stoch_loss(term_module, args):
    refs, key = args
    keys = jr.split(key, N_REFERENCES)
    def per(y0_i, k_i, r_i):
        ens = solve(term_module, y0_i,
                    jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Heun(),
                    args=forcing_smooth,
                    key=k_i, n_samples=M)
        return energy_score(ens, r_i, kernel=separation_distance, reduce="sum")
    return jnp.sum(jax.vmap(per)(y0s_ref, keys, refs))

solver_joint = optx.BFGS(rtol=1e-4, atol=1e-4)
sol_joint = optx.minimise(
    stoch_loss, solver=solver_joint, y0=stoch_init,
    args=(ref_trajs, jr.key(42)), 
)
beta_fit_joint = float(sol_joint.value.beta_w)
cs_fit_joint   = float(sol_joint.value.c_s)
print(f"converged in {int(sol_joint.stats['num_steps'])} steps  -> "
      f"beta_w = {beta_fit_joint * 100:.1f}%, "
      f"C_S = {cs_fit_joint:.3f}")
print(f"truth                  -> beta_w = {BETA_W_TRUE * 100:.1f}%  (no true C_S)")

stoch_fit = sol_joint.value

Animation -- the 10 deterministic reference trajectories drawn against the **smoothed** ocean current the model actually sees, each surrounded by a 30-member SDE ensemble drawn from the fitted parameters.


In [ ]:
ENS_SHOW = 30  # ensemble members per reference to draw

ens_show = jax.vmap(
    lambda y0_i, k_i: solve(stoch_fit, y0_i,
                             jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Heun(),
                             args=forcing_smooth,
                             key=k_i, n_samples=ENS_SHOW)
)(y0s_ref, jr.split(jr.key(7), N_REFERENCES))
ens_show_np = np.asarray(ens_show)        # (N_REFERENCES, ENS_SHOW, T, 2)
refs_np     = np.asarray(ref_trajs)       # (N_REFERENCES, T, 2)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

bg = ax.pcolormesh(LON, LAT, ocean_speed_smooth[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.7)
fig.colorbar(bg, ax=ax,
             label=r"$\| \mathbf{u}_o^{\mathrm{smooth}} \|$  (m s$^{-1}$)")
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.7, width=0.006, pivot="mid")

colors = plt.cm.tab10(np.arange(N_REFERENCES) % 10)
ens_lines = [[ax.plot([], [], color=colors[d], alpha=0.25, lw=0.5)[0]
              for _ in range(ENS_SHOW)]
             for d in range(N_REFERENCES)]
ref_lines = [ax.plot([], [], color=colors[d], lw=1.5)[0]
             for d in range(N_REFERENCES)]
ax.scatter(refs_np[:, 0, 1], refs_np[:, 0, 0],
           color="white", edgecolor="black", zorder=4, s=20, label="Starts")
ax.set_aspect("equal")
ax.set_xlim(float(LON.min()), float(LON.max()))
ax.set_ylim(float(LAT.min()), float(LAT.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
title = ax.set_title("")

def draw(k):
    field_k = k + 1
    bg.set_array(ocean_speed_smooth[field_k].ravel())
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    for d in range(N_REFERENCES):
        ref_lines[d].set_data(refs_np[d, : k + 1, 1], refs_np[d, : k + 1, 0])
        for j, ln in enumerate(ens_lines[d]):
            ln.set_data(ens_show_np[d, j, : k + 1, 1],
                        ens_show_np[d, j, : k + 1, 0])
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return [bg, q_w, title, *ref_lines, *[ln for grp in ens_lines for ln in grp]]

HTML(animation.FuncAnimation(fig, draw, frames=ens_show_np.shape[2],
                             interval=80, blit=False).to_jshtml())

## 4. More "advanced" modelisation

### 1. A stochastic ensemble via perturbed ODE

The SDE approach above hard-wires a *linear* noise model: the per-step displacement is $\mathbf{g}(t, \mathbf{X})\,\mathrm{d}\mathbf{W}$, a linear function of the Wiener
increment. 
More expressive stochastic simulators — such as some generative neural networks — learn a *non-linear* mapping $\mathbf{z} \mapsto \boldsymbol{\varepsilon}$ from a standard-normal seed to a richer, possibly multi-modal displacement distribution. 
The SDE solver cannot represent this: it always computes $\mathbf{g} \cdot \mathrm{d}\mathbf{W}$.

The ODE+controls pattern handles it directly. 
The user pre-samples a batch of noise trajectories (shape `(S, n_fine, 2)`) from any distribution and vmaps `solve` over them; each integration step receives one `ctrl` slice and the term applies whatever non-linear transform it likes. 
The solver never sees the noise — it simply multiplies the returned velocity by $\Delta t$.

Here we illustrate with a **tanh-squashed Gaussian**: `ctrl ~ N(0, I_2)` is mapped through $\tanh$ before being scaled by the local Smagorinsky amplitude $g$. 
Compared to the linear SDE this produces a *bounded* residual velocity (no arbitrarily large steps), while retaining the correct local scale. 
Replacing `tanh` with the generative model requires no changes to the solver or the integration loop.

In [ ]:
def perturbed_ode_term(t, y, args, ctrl):
    drift, g = smag_windage_term(t, y, args)
    # Non-linear transform: tanh squashes ctrl ~ N(0, I_2) into (-1, 1),
    # giving a bounded residual. Swap tanh for an neural network to get a
    # richer distribution — the solver and integration loop are unchanged.
    residual = g * jnp.tanh(ctrl) / jnp.sqrt(int_dt_sim)

    return drift + residual

S = 100  # ensemble size
# User owns the noise: pre-sample ctrl ~ N(0, I_2) for every member and step.
controls_batch = jr.normal(jr.key(2), shape=(S, n_save_sim, 2))

ens_perturbed = jax.vmap(
    lambda c: solve(perturbed_ode_term, y0,
                    jnp.array(t0_sim), n_save_sim, int_dt_sim, int_dt_sim, solver=Heun(),
                    args=(forcing_smooth, BETA_W_TRUE, CS), controls=c)
)(controls_batch)
print("perturbed ODE ensemble shape:", ens_perturbed.shape)

Animation — same as §3.1.

In [ ]:
ens_np = np.asarray(ens_perturbed)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

bg = ax.pcolormesh(LON, LAT, ocean_speed_smooth[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.7)
fig.colorbar(bg, ax=ax, label=r"$\| \mathbf{u}_o^{\mathrm{smooth}} \|$  (m s$^{-1}$)")
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.7, width=0.006,
                pivot="mid")

ens_lines = [ax.plot([], [], color="black", alpha=0.18, lw=0.6)[0]
             for _ in range(ens_np.shape[0])]
det_line, = ax.plot([], [], color="tab:orange", lw=2.0, label="Deterministic")
ax.scatter(y0[1], y0[0], color="white", edgecolor="black", zorder=4, label="Start")
ax.set_aspect("equal")
ax.set_xlim(float(LON.min()), float(LON.max()))
ax.set_ylim(float(LAT.min()), float(LAT.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
title = ax.set_title("")

def draw(k):
    field_k = k + 1
    bg.set_array(np.asarray(ocean_speed_smooth[field_k]).ravel())
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    for i, ln in enumerate(ens_lines):
        ln.set_data(ens_np[i, : k + 1, 1], ens_np[i, : k + 1, 0])
    det_line.set_data(traj_smooth_np[: k + 1, 1], traj_smooth_np[: k + 1, 0])
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return [bg, q_w, det_line, title, *ens_lines]

HTML(animation.FuncAnimation(fig, draw, frames=ens_np.shape[1], interval=80, blit=False).to_jshtml())


### 2. Surface-ocean Maxey–Riley: a full second-order inertial model

The windage term of §2 is an *ad-hoc* leeway model. 
The principled description of a finite-size buoyant object at the air–sea interface is the **Maxey–Riley equation**, adapted to the rotating ocean surface by [Beron-Vera, F., Olascoaga, M., & Miron, P. (2019, *Phys. Fluids*)](https://doi.org/10.1063/1.5110731). 
The particle carries its own velocity $\mathbf v_p$ which evolves as (their Eq. 29, on a $\beta$-plane):

$$
\frac{\mathrm d\mathbf v_p}{\mathrm dt}
 = \underbrace{R\,\frac{\mathrm D\mathbf v}{\mathrm Dt}}_{\text{fluid accel. + added mass}}
 + \underbrace{\tau^{-1}\,(\mathbf u - \mathbf v_p)}_{\text{drag to weighted forcing}}
 + \underbrace{R\!\left(f+\tfrac{\omega}{3}\right)\!\mathbf J\mathbf v
   - \left(f+\tfrac{R\omega}{3}\right)\!\mathbf J\mathbf v_p}_{\text{Coriolis + lift}},
\qquad
\frac{\mathrm d\mathbf x}{\mathrm dt}=\mathbf v_p .
$$

| symbol | meaning |
|---|---|
| $\mathbf v$ | seawater velocity (the field `u_o, v_o`); $\tfrac{\mathrm D\mathbf v}{\mathrm Dt}=\partial_t\mathbf v+(\mathbf v\!\cdot\!\nabla)\mathbf v$ |
| $\mathbf u=(1-\alpha)\,\mathbf v+\alpha\,\mathbf v_a$ | density-weighted forcing — seawater **and** air (`u_w, v_w`) velocities |
| $R=\frac{1-\Phi/2}{1-\Phi/6}$ | buoyancy parameter ($\Phi$ from the submerged fraction) |
| $\tau$ | inertial response (Stokes) time |
| $\alpha$ | air-drag weight, $\propto$ viscosity ratio and emerged fraction (small) |
| $f=2\Omega\sin\varphi$ | Coriolis parameter at latitude $\varphi$; $\omega=\partial_x v^2-\partial_y v^1$ vorticity |
| $\mathbf J$ | rotation by $90^\circ$, $\mathbf J(a,b)=(-b,a)$, i.e. $\mathbf k\times\,\cdot$ |

We can notice that as $\tau\to 0$ the particle collapses onto the *slow manifold* $\mathbf v_p\to\mathbf u$ (it rides the weighted forcing); finite $\tau$ is what makes floating objects deviate from passive tracers.

We integrate the model **in SI units** (velocity in m/s) and convert only the kinematic coupling $\dot{\mathbf x}=\mathbf v_p$ to deg/s. The material derivative and vorticity use grid-scale central differences of the (differentiable) bilinear interpolation. 
Following §3.1, we run this model — deterministic and stochastic alike — on the smoothed currents `forcing_smooth`.

In [ ]:
from typing import NamedTuple

OMEGA_EARTH = 7.2921e-5                                  # Earth angular rate (rad/s)
H_T = int_dt_sim / 2                                     # time step for d/dt (s)


class State(NamedTuple):
    x: jax.Array                                         # position [lat, lon]      (degrees)
    v: jax.Array                                         # velocity [east, north]   (m/s)


def _J(a):                                               # 90 deg rotation, i.e. k x a
    return jnp.array([-a[1], a[0]])


def _water(ds, t, la, lo):                               # seawater velocity [east, north] (m/s)
    return jnp.array([ds["u_o"].interp(t, la, lo), ds["v_o"].interp(t, la, lo)])


def _air(ds, t, la, lo):                                 # air velocity [east, north] (m/s)
    return jnp.array([ds["u_w"].interp(t, la, lo), ds["v_w"].interp(t, la, lo)])


def water_fields(ds, t, x):
    """Seawater velocity, material derivative Dv/Dt and vorticity at (t, x)."""
    lat, lon = x[0], x[1]
    w = _water(ds, t, lat, lon)
    dwdx = (_water(ds, t, lat, lon + DLON) - _water(ds, t, lat, lon - DLON)) / (2 * DX_M)
    dwdy = (_water(ds, t, lat + DLAT, lon) - _water(ds, t, lat - DLAT, lon)) / (2 * DY_M)
    dwdt = (_water(ds, t + H_T, lat, lon) - _water(ds, t - H_T, lat, lon)) / (2 * H_T)
    DvDt = dwdt + w[0] * dwdx + w[1] * dwdy              # advective (material) derivative
    omega = dwdx[1] - dwdy[0]                            # vertical vorticity
    return w, DvDt, omega


def maxey_riley_drift(t, y, args):
    ds, R, alpha, tau = args
    lat, lon = y.x[0], y.x[1]
    w, DvDt, omega = water_fields(ds, t, y.x)
    w_air = _air(ds, t, lat, lon)
    u_carrier = (1 - alpha) * w + alpha * w_air               # density-weighted carrier
    f = 2 * OMEGA_EARTH * jnp.sin(jnp.deg2rad(lat))           # Coriolis parameter

    dv = (R * DvDt                                            # fluid accel + added mass
          + (u_carrier - y.v) / tau                           # drag to carrier
          + R * (f + omega / 3) * _J(w)                       # Coriolis + lift (fluid)
          - (f + R * omega / 3) * _J(y.v))                    # Coriolis + lift (particle)
    dx = meters_to_degrees(jnp.array([y.v[1], y.v[0]]), lat)  # [north, east] -> deg/s
    return State(x=dx, v=dv)


# physical parameters for a buoyant, ~half-submerged float
R_BUOY = 0.6            # (1 - Phi/2)/(1 - Phi/6)
ALPHA = 0.05            # air-drag weight
TAU = 1 * 3600.0        # inertial response time (1 h; exaggerated so inertia is visible)

state0 = State(x=y0, v=jnp.zeros(2))                    # released at rest

# Substep (int_dt < save_dt) for stability of the stiff drag under an explicit solver.
traj_mr = solve(maxey_riley_drift, state0, jnp.array(t0_sim),
                n_save_sim, int_dt_sim / 2, int_dt_sim, solver=Tsit5(),
                args=(forcing_smooth, R_BUOY, ALPHA, TAU))


# Slow-manifold (tau -> 0) reference: a tracer riding the weighted carrier.
def carrier_term(t, y, args):
    ds, alpha = args
    la, lo = y[0], y[1]
    w = _water(ds, t, la, lo)
    w_air = _air(ds, t, la, lo)
    u_c = (1 - alpha) * w + alpha * w_air
    return meters_to_degrees(jnp.array([u_c[1], u_c[0]]), la)


traj_carrier = solve(carrier_term, y0, jnp.array(t0_sim),
                     n_save_sim, int_dt_sim / 2, int_dt_sim, solver=Tsit5(),
                     args=(forcing_smooth, ALPHA))
print("Full Maxey-Riley state:", traj_mr.x.shape, "+ velocity", traj_mr.v.shape)
print("Slow Manifold (tau->0):", traj_carrier.shape)

In [ ]:
# Inertia + Coriolis make the float deviate from the carrier it is dragged toward.
ts_traj = jnp.array(t0_sim) + jnp.arange(n_save_sim + 1) * int_dt_sim
hours = (ts_traj - ts_traj[0]) / 3600.0

vp_ms = jnp.sqrt((traj_mr.v ** 2).sum(-1))                         # |v_p| in m/s
u_c = jax.vmap(lambda t, x: (1 - ALPHA) * _water(forcing_smooth, t, x[0], x[1])
               + ALPHA * _air(forcing_smooth, t, x[0], x[1]))(
    ts_traj, traj_mr.x)
uc_ms = jnp.sqrt((u_c ** 2).sum(-1))

mr, ca = np.asarray(traj_mr.x), np.asarray(traj_carrier)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6))
axL.pcolormesh(LON, LAT, ocean_speed_smooth[1], cmap=cmocean.cm.speed, vmin=0,
               vmax=float(ocean_speed_smooth.max()), shading="auto")
axL.plot(ca[:, 1], ca[:, 0], color="tab:blue", lw=2.0, label="Slow Manifold")
axL.plot(mr[:, 1], mr[:, 0], color="tab:orange", lw=2.0, label="Maxey-Riley")
axL.scatter(*ca[0, ::-1], color="white", edgecolor="black", zorder=3, label="Release")
axL.set_aspect("equal"); axL.set_xlabel("lon"); axL.set_ylabel("lat")
axL.legend(loc="upper left", fontsize=8)

axR.plot(hours, uc_ms, color="tab:blue", lw=2, label="Slow Manifold")
axR.plot(hours, vp_ms, color="tab:orange", lw=2, label="Maxey-Riley")
axR.set_xlabel("time (h)"); axR.set_ylabel("speed (m/s)")
axR.legend(fontsize=8)
fig.tight_layout()

The §3.1 ensemble injected the unresolved scales as a Smagorinsky diffusion *on position*, with a diffusivity $K$ modelled from the local strain and amplitude $\sigma=\sqrt{2K}$. 
The Maxey–Riley model lets us be more physical. 
A real float is dragged through **unresolved turbulence**, so the seawater velocity felt in the drag is $\mathbf v + \mathbf v'$, with
$\mathbf v'$ a sub-grid fluctuation; modelling $\mathbf v'$ as white-in-time turns the drag into a stochastic forcing *on the velocity*. 
Here $K$ is the ocean eddy diffusivity with $K=\sigma_v^2\,\tau$, and the drag makes the slip an Ornstein–Uhlenbeck process whose
stationary variance is $\sigma_v^2=\sigma^2\tau/2$; hence
$$
\sigma=\frac{\sqrt{2K}}{\tau}
$$
-- the same $\sqrt{2K}$ as in §3.1, now divided by the response time $\tau$ because the noise drives the velocity rather than the position directly.

Lateral stirring is **anisotropic** — stronger *along* the flow than across it (shear dispersion), so $K_\parallel>K_\perp$. 
With $\hat{\mathbf e}_\parallel$ along the local seawater velocity, the diffusion maps a 2-D Wiener increment into the velocity leaf,
$$
\Sigma\,\mathrm d\mathbf W = \frac{\sqrt{2K_\parallel}}{\tau}\,\mathrm dW_\parallel\,\hat{\mathbf e}_\parallel + \frac{\sqrt{2K_\perp}}{\tau}\,\mathrm dW_\perp\,\hat{\mathbf e}_\perp ,
$$
a state-dependent, non-diagonal $2\times2$ operator. We express it as a `lineax.FunctionLinearOperator` returning a `State` tangent (noise on $\mathbf v_p$ only) and declare the 2-D Brownian space with `brownian_structure`.

> **The diffusivity is driven by the §3.1 Smagorinsky closure.** Rather than prescribe $K_\parallel, K_\perp$ as constants, we set them from the §3.1 Smagorinsky strain, $K = C_S\,\Delta x^2\,\lVert S\rVert$, evaluated from the local current with `Dataset.neighborhood` -- only the *injection* differs (on the velocity $\mathbf v_p$ here, on position in §3.1). Anisotropy is then retained with a **vector-valued coefficient** $C_S=(C_\parallel, C_\perp)$: $K_\parallel = C_\parallel\,\Delta x^2\,\lVert S\rVert$ and $K_\perp = C_\perp\,\Delta x^2\,\lVert S\rVert$ share the strain magnitude but split the along/cross-stream amplitude, so the noise strengthens in high-strain regions (fronts, eddy rims) while the $C_\parallel/C_\perp$ ratio fixes the anisotropy. A fuller variant replaces the velocity frame $(\hat{\mathbf e}_\parallel,\hat{\mathbf e}_\perp)$ by the eigenframe of the strain-rate tensor $S$, letting the deformation set both the magnitude and the orientation of the noise.

In [ ]:
import lineax as lx
from pastax import EulerHeun

NOISE_DIM = jax.ShapeDtypeStruct((2,), jnp.float64)   # 2-D Wiener (along/cross-stream)
C_PAR = 2.0     # along-flow Smagorinsky coefficient
C_PERP = 0.4    # cross-flow Smagorinsky coefficient


def maxey_riley_sde(t, y, args):
    ds, R, alpha, tau, c_par, c_perp = args
    drift = maxey_riley_drift(t, y, (ds, R, alpha, tau))

    w, _, _ = water_fields(ds, t, y.x)
    speed = safe_sqrt(w[0] ** 2 + w[1] ** 2)
    inv = 1.0 / (speed + 1e-9)
    e_par = w * inv                              # along-flow unit vector
    e_perp = jnp.array([-w[1], w[0]]) * inv      # cross-flow unit vector

    # Smagorinsky strain |S| from a 3x3 patch of the smoothed current (cf. §3.1),
    # split anisotropically by a vector-valued C_S = (c_par, c_perp).
    patches = ds.neighborhood(t, y.x[0], y.x[1], t_window=0, lat_window=1, lon_window=1)
    u_patch = patches["u_o"][0]
    v_patch = patches["v_o"][0]
    du_dx = (u_patch[1, 2] - u_patch[1, 0]) / (2 * DX_M)
    du_dy = (u_patch[2, 1] - u_patch[0, 1]) / (2 * DY_M)
    dv_dx = (v_patch[1, 2] - v_patch[1, 0]) / (2 * DX_M)
    dv_dy = (v_patch[2, 1] - v_patch[0, 1]) / (2 * DY_M)
    strain = safe_sqrt(2 * du_dx ** 2 + 2 * dv_dy ** 2 + (du_dy + dv_dx) ** 2)
    k_par = c_par * DX_M ** 2 * strain           # along-flow eddy diffusivity (m^2/s)
    k_perp = c_perp * DX_M ** 2 * strain         # cross-flow eddy diffusivity (m^2/s)

    # velocity-noise amplitude: sigma = sqrt(2K)/tau has units m*s^-3/2, so
    # sigma*dW (dW ~ sqrt(dt)) is a velocity increment (m/s) -> diffusivity K.
    sig_par = safe_sqrt(2 * k_par) / tau
    sig_perp = safe_sqrt(2 * k_perp) / tau
    Sigma = jnp.stack([sig_par * e_par, sig_perp * e_perp], axis=1)   # (2, 2)

    diffusion = lx.FunctionLinearOperator(
        lambda dW: State(x=jnp.zeros(2), v=Sigma @ dW), NOISE_DIM)
    return drift, diffusion


mr_ens = solve(maxey_riley_sde, state0, jnp.array(t0_sim),
               n_save_sim, int_dt_sim / 2, int_dt_sim, solver=EulerHeun(),
               args=(forcing_smooth, R_BUOY, ALPHA, TAU, C_PAR, C_PERP),
               key=jr.key(0), n_samples=100, brownian_structure=NOISE_DIM)
print("Maxey-Riley ensemble:", mr_ens.x.shape)       # (100, n_save + 1, 2)

Animation — the stochastic Maxey–Riley ensemble (thin black) spreads anisotropically along the flow under the eddy-diffusivity noise, around the deterministic inertial path (orange), over the evolving ocean speed and wind.

In [ ]:
# Animation — stochastic Maxey-Riley ensemble over the time-evolving forcing.
ens_np = np.asarray(mr_ens.x)
mr = np.asarray(traj_mr.x); ca = np.asarray(traj_carrier)

fig, ax = plt.subplots(figsize=(5, 5))
plt.close(fig)

bg = ax.pcolormesh(LON, LAT, ocean_speed_smooth[0], cmap=cmocean.cm.speed,
                   vmin=0, vmax=ocean_clim, alpha=0.7)
fig.colorbar(bg, ax=ax, label=r"$\| \mathbf{u}_o^{\text{smooth}} \|$  (m s$^{-1}$)")
q_w = ax.quiver(LON[::20], LAT[::20], u_w[0][::20, ::20], v_w[0][::20, ::20],
                scale=120, color="black", alpha=0.7, width=0.006, pivot="mid")

ens_lines = [ax.plot([], [], color="black", alpha=0.15, lw=0.6)[0]
             for _ in range(ens_np.shape[0])]
mr_line, = ax.plot([], [], color="tab:orange", lw=2.0, label="Deterministic Maxey-Riley")
ax.scatter(y0[1], y0[0], color="white", edgecolor="black", zorder=4, label="Release")
ax.set_aspect("equal")
ax.set_xlim(float(LON.min()), float(LON.max()))
ax.set_ylim(float(LAT.min()), float(LAT.max()))
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.legend(loc="upper right")
title = ax.set_title("")


def draw(k):
    field_k = k + 1
    bg.set_array(np.asarray(ocean_speed_smooth[field_k]).ravel())
    q_w.set_UVC(np.asarray(u_w[k][::20, ::20]), np.asarray(v_w[k][::20, ::20]))
    for i, ln in enumerate(ens_lines):
        ln.set_data(ens_np[i, : k + 1, 1], ens_np[i, : k + 1, 0])
    mr_line.set_data(mr[: k + 1, 1], mr[: k + 1, 0])
    title.set_text(f"$t$ = {ts_sim[k] / 3600:.0f} h")
    return [bg, q_w, mr_line, title, *ens_lines]


HTML(animation.FuncAnimation(fig, draw, frames=ens_np.shape[1],
                             interval=80, blit=False).to_jshtml())

This capstone packs the whole framework into one physically-grounded model: a **PyTree state** $(\mathbf x,\mathbf v_p)$ for the second-order Maxey–Riley dynamics (fluid acceleration, Coriolis, lift and drag to a wind/water-weighted carrier); an **operator-valued diffusion** (`lineax.FunctionLinearOperator`) for anisotropic, flow-aligned turbulence that a diagonal `g` cannot represent; and a **`browian_structure`** decoupling the 2-D Wiener space from the 4-D state. 
Because the term is built from differentiable `interp` calls, the whole simulator is end-to-end differentiable — the parameters $(R,\alpha,\tau,C_\parallel,C_\perp)$ can be **learned** with the scoring rules of §3.2.

## Where to next

- See the [API reference](api.md) for the full surface of `solve`, `Dataset`, and the interpolation, 
  metric, and score helpers.
- Replace `Dataset.from_arrays` with `Dataset.from_xarray` to drive the simulator from
  real zarr or netCDF forcing fields.
- The `solve` integrator supports both reverse-mode (`jax.grad`, used by `BFGS` above) and
  forward-mode (`jax.jvp`, used by `Levenberg-Marquardt` above) auto-differentiation.
  Reverse mode is the low-memory default (`adjoint="checkpointed"`); forward mode requires
  `solve(..., adjoint="forward")`, as in the §2.2 fit.